# SimSat DiLoCo Round 0 Learner — v5 (diagnostic, OOM-safe)

v4 OOMed in `prepare_model_for_kbit_training` on T4 (fp32 cast on Gemma-4 norms + embeddings exceeds 14.5 GiB). v5 drops that fix attempt and ships the LoRA-state instrumentation alone, plus an explicit `model.train()` call (suspected cause: if the model is still in eval mode after `_evaluate_loss`, PEFT may bypass LoRA in the forward).

Three snapshots: INIT (before train), POST (after train, in-memory), SAVED (read back from disk). Verdict logic is in the final cell block.

Also writes a verbose diagnostic JSON to `/kaggle/working/diloco_continued_adapter/v5_diagnostic.json` so the next-session can read it without parsing the kernel log.

In [ ]:
import hashlib, json, os, shutil, sys
from pathlib import Path

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')

src_marker = next(INPUT.rglob('continue_gemma4_adapter.py'))
SRC = src_marker.parents[1]
print(f'SRC: {SRC}')

READ_ONLY = next(INPUT.rglob('adapter_config.json')).parent
ADAPTER = WORK / 'global_adapter_writable'
if ADAPTER.exists():
    shutil.rmtree(ADAPTER)
shutil.copytree(READ_ONLY, ADAPTER)
cfg_path = ADAPTER / 'adapter_config.json'
cfg = json.loads(cfg_path.read_text())
prev_mode = cfg.get('inference_mode')
cfg['inference_mode'] = False
cfg_path.write_text(json.dumps(cfg, indent=2))
print(f'Adapter staged: {ADAPTER}  (inference_mode: {prev_mode!r} -> False)')

def _sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

SEED_HASH = _sha256(ADAPTER / 'adapter_model.safetensors')
print(f'Seed sha256: {SEED_HASH}')

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
sys.path.insert(0, str(SRC / 'kaggle'))
sys.path.insert(0, str(SRC))

import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'unsloth', 'unsloth_zoo', 'torchao'], check=False)
shutil.rmtree('/kaggle/working/unsloth_compiled_cache', ignore_errors=True)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'transformers>=4.51.0', 'peft>=0.12.0', 'accelerate>=0.33.0',
    'bitsandbytes>=0.44.0', 'datasets>=2.19.0', 'safetensors>=0.4.3'])

import continue_gemma4_adapter as runner
import torch
from transformers import AutoTokenizer

MODEL_ID = '/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1'
TRAIN_PATH = next(Path('/kaggle/input').rglob('simsat_train.jsonl'))
print(f'MODEL: {MODEL_ID}')
print(f'TRAIN: {TRAIN_PATH}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

rows = runner._load_jsonl(str(TRAIN_PATH))
dataset = runner.ChatMLDataset(rows, tokenizer, 1024)
print(f'dataset: {len(dataset)} rows | tokens={dataset.total_tokens:,} supervised={dataset.supervised_tokens:,}')

base = runner._load_base_model(MODEL_ID)
model = runner._attach_trainable_adapter(base, str(ADAPTER))
model.train()  # explicit; runner's _evaluate_loss leaves it in train at the end but be safe
print(f'model.training: {model.training}')

def _lora_state(model):
    out = {}
    for name, param in model.named_parameters():
        if 'lora_' not in name:
            continue
        t = param.detach().cpu().float().numpy()
        h = hashlib.sha256(t.tobytes()).hexdigest()
        out[name] = (h, float(abs(t).sum()), float(t.std()))
    return out

INIT = _lora_state(model)
print(f'INIT lora params: {len(INIT)}')
for k in list(INIT)[:3]:
    h, s, sd = INIT[k]
    print(f'  {k}  hash={h[:16]}  abs_sum={s:.4f}  std={sd:.6f}')

pad_id = int(tokenizer.pad_token_id or tokenizer.eos_token_id)
loss_before = runner._evaluate_loss(model, dataset, pad_id, 16)
print(f'loss_before: {loss_before}')
model.train()  # _evaluate_loss flips back to train but be explicit
print(f'model.training (after eval_before, before train loop): {model.training}')

# Quick mid-train sanity: capture grad after first backward to see if LoRA gets gradients
from torch.utils.data import DataLoader
loader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=lambda b: runner._collate(b, pad_id))
first_batch = next(iter(loader))
first_batch = {k: v.to(next(model.parameters()).device) for k, v in first_batch.items()}
out = model(**first_batch)
out.loss.backward()
lora_grads_present = 0
lora_grads_nonzero = 0
lora_grad_max = 0.0
for name, p in model.named_parameters():
    if 'lora_' not in name:
        continue
    if p.grad is not None:
        lora_grads_present += 1
        nz = float(p.grad.abs().max())
        if nz > 0:
            lora_grads_nonzero += 1
        lora_grad_max = max(lora_grad_max, nz)
print(f'After 1 backward: lora params with .grad: {lora_grads_present}, with non-zero grad: {lora_grads_nonzero}, max |grad|={lora_grad_max:.6e}')
model.zero_grad(set_to_none=True)

steps = runner._train(model, dataset, pad_token_id=pad_id, batch_size=1, grad_accum=8,
                       max_steps=120, epochs=1, lr=5e-5, warmup_steps=10, seed=42)
loss_after = runner._evaluate_loss(model, dataset, pad_id, 16)
print(f'loss_after: {loss_after}')

POST = _lora_state(model)
changed = sum(1 for k in INIT if INIT[k][0] != POST[k][0])
print(f'IN-MEMORY: {changed}/{len(INIT)} lora params changed')
for k in list(POST)[:3]:
    h, s, sd = POST[k]
    delta = abs(s - INIT[k][1])
    print(f'  {k}  hash={h[:16]}  abs_sum={s:.4f}  delta_abs_sum={delta:.6f}')

OUT = WORK / 'diloco_continued_adapter'
OUT.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(OUT))
tokenizer.save_pretrained(str(OUT))
OUT_HASH = _sha256(OUT / 'adapter_model.safetensors')
print(f'Continued sha256: {OUT_HASH}')

from safetensors import safe_open
SAVED = {}
with safe_open(OUT / 'adapter_model.safetensors', framework='numpy') as f:
    for k in f.keys():
        if 'lora_' in k:
            t = f.get_tensor(k)
            SAVED[k] = (hashlib.sha256(t.tobytes()).hexdigest(), float(abs(t).sum()))
print(f'SAVED lora params: {len(SAVED)}')

print()
print('=== DIAGNOSIS ===')
print(f'INIT vs POST changed: {changed}/{len(INIT)}')
print(f'lora_grads_nonzero after first backward: {lora_grads_nonzero}/{lora_grads_present}')
print(f'Continued sha256 == seed sha256: {OUT_HASH == SEED_HASH}')
if lora_grads_nonzero == 0:
    verdict = 'GRADIENTS_NEVER_REACH_LORA — backward pass routes around LoRA modules entirely'
elif changed == 0:
    verdict = 'GRADS_PRESENT_BUT_OPTIMIZER_NOT_UPDATING — optimizer wraps wrong tensor refs'
elif OUT_HASH == SEED_HASH:
    verdict = 'TRAIN_OK_SAVE_BROKEN — save_pretrained writes load-time weights instead of trained ones'
else:
    verdict = 'TRAIN_OK_SAVE_OK — pipeline works, eval should now show v10 deltas'
print(f'VERDICT: {verdict}')

diag = {
    'lora_param_count': len(INIT),
    'lora_grads_present_after_first_backward': lora_grads_present,
    'lora_grads_nonzero_after_first_backward': lora_grads_nonzero,
    'lora_grad_max_after_first_backward': lora_grad_max,
    'lora_changed_in_memory': changed,
    'loss_before': loss_before,
    'loss_after': loss_after,
    'seed_sha256': SEED_HASH,
    'continued_sha256': OUT_HASH,
    'verdict': verdict,
    'sample_diffs': [
        {'name': k,
         'init_hash': INIT[k][0][:16], 'post_hash': POST[k][0][:16],
         'init_abs_sum': INIT[k][1], 'post_abs_sum': POST[k][1],
         'delta_abs_sum': abs(POST[k][1] - INIT[k][1])}
        for k in list(INIT)[:5]
    ],
}
(OUT / 'v5_diagnostic.json').write_text(json.dumps(diag, indent=2))
print(f'\ndiagnostic: {OUT / "v5_diagnostic.json"}')